In [1]:
print(4)

4


In [2]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [4]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [5]:
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

In [8]:

from rag_helper import RagHelper


class RAGVector(RagHelper):

    def __init__(self, embedder,course='llm-zoomcamp', **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder
        self.course = course

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [11]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
    # model="nvidia/nemotron-3-ultra-550b-a55b:free",
    model="nvidia/nemotron-3-super-120b-a12b:free",

)